# Agentic RAG: query planning and multi-hop reasoning

The single-shot RAG bot you built in Lab 02 silently fails on questions like "what was the revenue impact of the migration we did in Q3 last year compared to Q3 the year before?" The answer needs two separate retrievals plus a calculation, and the bot's single-vector query collapses both into a midpoint embedding that retrieves neither.

You have 90 minutes to build an agentic RAG pipeline that:

1. Decomposes the user query into 2 to 5 sub-queries via Claude Sonnet 4.6 tool calling (planner).
2. Executes each sub-query against the Lab 02 hybrid retrieval pipeline (executor).
3. Composes a final answer grounded in retrieved chunks, citing each claim by chunk ID (composer).
4. Refuses to answer when retrieval returns nothing relevant (the "I don't know" pattern; graded).

The retrieval substrate (OpenSearch Serverless + Supabase pgvector) is re-provisioned at setup, same shape as Lab 02. Same `labrun-agentic-rag-query-planning-` prefix on AWS resources. Same 2-hour watchdog Lambda that destroys the collection if cleanup never runs.

Estimated time: 90 minutes. Session window: 120 minutes.

Cost: about $1.80 to $2.50 on a clean first run. Claude Sonnet does the planning and composition (biggest line item); OpenSearch Serverless is the same hourly bill as Lab 02; embeddings and pgvector are pennies. A debugging session that chases a tool-call format bug across five planner re-runs lands around $3.00 to $3.50.

Two bucks. The killer moment is the executor trace JSON showing three distinct retrievals collapsing into one composed answer with chunk citations.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. labrun-checks 0.7.0 stays consistent with the rest of
# the AI/ML engineering track. opensearch-py and requests-aws4auth talk to
# OpenSearch Serverless; cohere is optional (skip-by-default reranker).

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 openai==1.54.4 boto3==1.35.71 opensearch-py==2.7.1 requests-aws4auth==1.3.1 supabase==2.10.0 psycopg2-binary==2.9.10 pydantic==2.9.2 cohere==5.13.3 requests==2.32.3

In [ ]:
# Imports and per-lab constants. AWS resources carry the
# labrun-agentic-rag-query-planning- prefix; the pgvector table uses the
# underscored form because Postgres dislikes hyphens in unquoted identifiers.

import atexit
import getpass
import io
import json
import os
import re
import sys
import time
import uuid
import zipfile
from typing import Any, Optional

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "agentic-rag-query-planning"
LAB_TAG_VALUE = LAB_ID
RESOURCE_PREFIX = "labrun-agentic-rag-query-planning-"
PGVECTOR_TABLE = "labrun_agentic_rag_query_planning_chunks"

REGION = "us-east-1"
COLLECTION_NAME = RESOURCE_PREFIX + "collection"
INDEX_NAME = "chunks"
ENCRYPTION_POLICY_NAME = "labrun-arqp-enc"
NETWORK_POLICY_NAME = "labrun-arqp-net"
DATA_ACCESS_POLICY_NAME = "labrun-arqp-data"
WATCHDOG_RULE_NAME = RESOURCE_PREFIX + "cleanup-rule"
WATCHDOG_LAMBDA_NAME = RESOURCE_PREFIX + "cleanup-fn"
WATCHDOG_ROLE_NAME = RESOURCE_PREFIX + "watchdog-role"

TRACE_PATH = "/content/agent_traces.json"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

PLANNER_MODEL = "claude-sonnet-4-6"
COMPOSER_MODEL = "claude-sonnet-4-6"
PING_MODEL = "claude-haiku-4-5"

# Cosine similarity threshold below which the agent refuses to answer.
# Stored as a constant so the validator can reference the same number.
REFUSAL_THRESHOLD = 0.3

CANONICAL_REFUSAL_PREFIX = (
    "I don't have enough information in the available documents to answer "
    "this confidently. Here is what I found"
)

# Eval question that exercises the multi-hop path. Used by Checkpoints 2, 3, 4.
TEST_QUERY = (
    "Compare the revenue impact of the Q1 2024 migration against the Q2 2024 "
    "migration, and tell me which quarter delivered more revenue."
)

# 5 deliberately-impossible queries. The setup corpus contains nothing about
# any of these subjects, so every sub-query should land below REFUSAL_THRESHOLD.
IMPOSSIBLE_QUERIES = [
    "What did our 1958 board meeting in Reykjavik decide about Antarctic operations?",
    "How many ostriches did the marketing team adopt during the Crimean expansion?",
    "Which submarine pilot signed off on the Q4 2031 lunar revenue plan?",
    "What was the verdict in the puffin v. iguana copyright case the legal team handled?",
    "Which dragonfly species did our supply chain team breed for the 1623 logistics review?",
]


In [ ]:
# NBVAL_SKIP
# Bring-your-own-keys. None hit disk. Each credential gets a cheap validation
# before register() so a typo surfaces here, not 30 minutes into the lab.

import anthropic
from openai import OpenAI
from supabase import create_client
import psycopg2

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://xxx.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
DB_URI = getpass.getpass(
    "Supabase Postgres connection string "
    "(Project Settings > Database > Connection string > URI; pooled URL recommended): "
).strip()
COHERE_API_KEY = getpass.getpass("COHERE_API_KEY (optional, press Enter to skip): ").strip()

required = {
    "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY,
    "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY_ID,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "DB_URI": DB_URI,
}
missing = [k for k, v in required.items() if not v]
if missing:
    print("Missing required credentials: " + ", ".join(missing))
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Cheap Anthropic ping on Haiku (a fraction of a cent).
try:
    _anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = _anthropic_client.messages.create(
        model=PING_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

# Cheap OpenAI ping (models.list is free).
try:
    _openai_client = OpenAI(api_key=OPENAI_API_KEY)
    _ = list(_openai_client.models.list())[:1]
    print("OpenAI credentials validated.")
except Exception as exc:
    print(f"OpenAI key rejected: {exc!r}")
    raise SystemExit(1)

# STS GetCallerIdentity.
try:
    sts = boto3.client("sts", region_name=REGION)
    identity = sts.get_caller_identity()
    ACCOUNT_ID = identity["Account"]
    print(f"AWS credentials validated. Account: {ACCOUNT_ID}, Region: {REGION}")
except Exception as exc:
    print(f"AWS credentials rejected: {exc!r}")
    print("Check AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY for the labrun-test IAM user.")
    raise SystemExit(1)

# Supabase REST client + direct Postgres connection.
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
try:
    _conn = psycopg2.connect(DB_URI)
    _conn.autocommit = True
    with _conn.cursor() as _cur:
        _cur.execute("SELECT 1")
    _conn.close()
    print("Supabase Postgres connection validated.")
except Exception as exc:
    print(f"Postgres connection failed: {exc!r}")
    raise SystemExit(1)

SESSION_CREDENTIALS = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region": REGION,
    "account_id": ACCOUNT_ID,
    "supabase_url": SUPABASE_URL,
    "service_role_key": SUPABASE_SERVICE_ROLE_KEY,
    "db_uri": DB_URI,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=SESSION_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom adapter, atexit safety net, orphan scan. Critical
# tier (OpenSearch collection + its policies) tears down first per
# RESOURCE-SAFETY-SPEC Section 2 Rule 2. The 2-hour watchdog (EventBridge +
# Lambda + IAM role) is standard tier but still in the manifest so a clean
# cleanup removes it before the watchdog ever fires.

CLEANUP_MANIFEST = [
    # Critical tier: OpenSearch Serverless ($0.96/OCU-hour) tears down first.
    CleanupResource(
        type="aoss_collection",
        id=COLLECTION_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=(
            f"aws opensearchserverless delete-collection --id <COLLECTION_ID> --region {REGION}"
        ),
    ),
    CleanupResource(
        type="aoss_security_policy_encryption",
        id=ENCRYPTION_POLICY_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws opensearchserverless delete-security-policy "
            f"--name {ENCRYPTION_POLICY_NAME} --type encryption --region {REGION}"
        ),
    ),
    CleanupResource(
        type="aoss_security_policy_network",
        id=NETWORK_POLICY_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws opensearchserverless delete-security-policy "
            f"--name {NETWORK_POLICY_NAME} --type network --region {REGION}"
        ),
    ),
    CleanupResource(
        type="aoss_access_policy",
        id=DATA_ACCESS_POLICY_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws opensearchserverless delete-access-policy "
            f"--name {DATA_ACCESS_POLICY_NAME} --type data --region {REGION}"
        ),
    ),
    # Standard tier: 2-hour watchdog wiring.
    CleanupResource(
        type="eventbridge_rule",
        id=WATCHDOG_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {WATCHDOG_RULE_NAME} --ids 1 --region {REGION} && "
            f"aws events delete-rule --name {WATCHDOG_RULE_NAME} --region {REGION}"
        ),
    ),
    CleanupResource(
        type="lambda_function",
        id=WATCHDOG_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {WATCHDOG_LAMBDA_NAME} --region {REGION}"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=WATCHDOG_ROLE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws iam delete-role --role-name {WATCHDOG_ROLE_NAME}"
        ),
    ),
    # Standard tier: Supabase pgvector + local trace file.
    CleanupResource(
        type="supabase_table",
        id=PGVECTOR_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{PGVECTOR_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="local_file",
        id=TRACE_PATH,
        region="local",
        cli_delete_command=f"rm -f {TRACE_PATH}",
    ),
]


class AgenticRagCleanupAdapter:
    """Hand-rolled cleanup adapter for OpenSearch Serverless + Supabase pgvector
    + EventBridge/Lambda watchdog + local trace file.

    TODO: migrate to labrun-checks adapters once shipped. The Lab 02 hybrid lab
    also carries this same adapter; the two converge in the next labrun-checks
    release that ships aoss + vector_db adapter modules.
    """

    def __init__(self, credentials):
        self._creds = credentials
        self._aoss = boto3.client(
            "opensearchserverless",
            region_name=credentials["region"],
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
        )
        self._events = boto3.client(
            "events",
            region_name=credentials["region"],
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
        )
        self._lambda = boto3.client(
            "lambda",
            region_name=credentials["region"],
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
        )
        self._iam = boto3.client(
            "iam",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
        )
        self._tagging = boto3.client(
            "resourcegroupstaggingapi",
            region_name=credentials["region"],
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
        )
        self._db_uri = credentials.get("db_uri")

    def _pg_exec(self, sql):
        conn = psycopg2.connect(self._db_uri)
        conn.autocommit = True
        try:
            with conn.cursor() as cur:
                cur.execute(sql)
        finally:
            conn.close()

    def _pg_table_exists(self, table_name):
        conn = psycopg2.connect(self._db_uri)
        try:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT 1 FROM information_schema.tables "
                    "WHERE table_schema = 'public' AND table_name = %s",
                    (table_name,),
                )
                return cur.fetchone() is not None
        finally:
            conn.close()

    def _find_collection_id(self, name):
        resp = self._aoss.list_collections(collectionFilters={"name": name})
        for s in resp.get("collectionSummaries", []):
            if s.get("name") == name:
                return s.get("id")
        return None

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        try:
            if rtype == "aoss_collection":
                cid = self._find_collection_id(rid)
                if cid:
                    self._aoss.delete_collection(id=cid)
                    # Poll for DELETED for up to 5 minutes.
                    for _ in range(60):
                        try:
                            details = self._aoss.batch_get_collection(ids=[cid])
                            if not details.get("collectionDetails"):
                                break
                            status = details["collectionDetails"][0].get("status")
                            if status == "DELETED":
                                break
                        except ClientError:
                            break
                        time.sleep(5)
            elif rtype == "aoss_security_policy_encryption":
                try:
                    self._aoss.delete_security_policy(name=rid, type="encryption")
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("ResourceNotFoundException",):
                        raise
            elif rtype == "aoss_security_policy_network":
                try:
                    self._aoss.delete_security_policy(name=rid, type="network")
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("ResourceNotFoundException",):
                        raise
            elif rtype == "aoss_access_policy":
                try:
                    self._aoss.delete_access_policy(name=rid, type="data")
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("ResourceNotFoundException",):
                        raise
            elif rtype == "eventbridge_rule":
                try:
                    self._events.remove_targets(Rule=rid, Ids=["1"], Force=True)
                except ClientError:
                    pass
                try:
                    self._events.delete_rule(Name=rid, Force=True)
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("ResourceNotFoundException",):
                        raise
            elif rtype == "lambda_function":
                try:
                    self._lambda.delete_function(FunctionName=rid)
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("ResourceNotFoundException",):
                        raise
            elif rtype == "iam_role":
                # Detach attached policies first, delete inline policies, then the role.
                try:
                    attached = self._iam.list_attached_role_policies(RoleName=rid).get("AttachedPolicies", [])
                    for p in attached:
                        self._iam.detach_role_policy(RoleName=rid, PolicyArn=p["PolicyArn"])
                    inline = self._iam.list_role_policies(RoleName=rid).get("PolicyNames", [])
                    for pname in inline:
                        self._iam.delete_role_policy(RoleName=rid, PolicyName=pname)
                    self._iam.delete_role(RoleName=rid)
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("NoSuchEntity",):
                        raise
            elif rtype == "supabase_table":
                self._pg_exec(f"DROP TABLE IF EXISTS public.{rid} CASCADE")
            elif rtype == "local_file":
                try:
                    os.remove(rid)
                except FileNotFoundError:
                    pass
            else:
                raise ValueError(f"Unknown resource type {rtype!r}")
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("ResourceNotFoundException", "NoSuchEntity", "ResourceNotFoundFault"):
                return
            raise

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        try:
            if rtype == "aoss_collection":
                return self._find_collection_id(rid) is not None
            if rtype in ("aoss_security_policy_encryption", "aoss_security_policy_network"):
                policy_type = "encryption" if "encryption" in rtype else "network"
                try:
                    self._aoss.get_security_policy(name=rid, type=policy_type)
                    return True
                except ClientError:
                    return False
            if rtype == "aoss_access_policy":
                try:
                    self._aoss.get_access_policy(name=rid, type="data")
                    return True
                except ClientError:
                    return False
            if rtype == "eventbridge_rule":
                try:
                    self._events.describe_rule(Name=rid)
                    return True
                except ClientError:
                    return False
            if rtype == "lambda_function":
                try:
                    self._lambda.get_function(FunctionName=rid)
                    return True
                except ClientError:
                    return False
            if rtype == "iam_role":
                try:
                    self._iam.get_role(RoleName=rid)
                    return True
                except ClientError:
                    return False
            if rtype == "supabase_table":
                try:
                    return self._pg_table_exists(rid)
                except Exception:
                    return False
            if rtype == "local_file":
                return os.path.exists(rid)
        except Exception:
            return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # Compose orphans from AWS tag scan and Supabase prefix scan.
        arns = []
        try:
            paginator = self._tagging.get_paginator("get_resources")
            for page in paginator.paginate(
                TagFilters=[{"Key": "labrun:lab-slug", "Values": [lab_slug]}]
            ):
                for r in page.get("ResourceTagMappingList", []):
                    arns.append(r["ResourceARN"])
        except Exception:
            pass
        # Supabase prefix scan: any table with the lab's underscore prefix.
        try:
            conn = psycopg2.connect(self._db_uri)
            try:
                with conn.cursor() as cur:
                    cur.execute(
                        "SELECT table_name FROM information_schema.tables "
                        "WHERE table_schema = 'public' AND table_name LIKE %s",
                        ("labrun_agentic_rag_query_planning_%",),
                    )
                    arns.extend([row[0] for row in cur.fetchall()])
            finally:
                conn.close()
        except Exception:
            pass
        return arns


CLEANUP_ADAPTER = AgenticRagCleanupAdapter(credentials=SESSION_CREDENTIALS)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(SESSION_CREDENTIALS, LAB_TAG_VALUE, REGION)


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing labrun-agentic-rag-query-planning resources detected:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision resources for this session.")


## Seed the corpus

The corpus is synthetic. Twelve documents with `quarter=Q1 2024` style temporal markers so that a multi-hop question like "compare Q1 2024 migration revenue to Q2 2024 migration revenue" has two distinct retrievals to make. Deterministic content (no randomness, no LLM generation) so the lab's eval is reproducible.

The next cell loads the corpus into memory. The cell after that provisions the retrieval substrate and indexes every chunk.

In [ ]:
# Synthetic corpus. Each entry is a chunk with integer id, text, and
# quarter metadata. Twelve chunks across four quarters; six "migration" docs
# (two per migration quarter pair), three "platform" docs, three filler docs.

CORPUS = [
    {"id": 1, "quarter": "Q1 2024", "topic": "migration",
     "text": "Migration project alpha completed in Q1 2024 delivered $4.2M in incremental revenue. Cutover from the legacy oracle warehouse to the modern lakehouse landed two weeks ahead of plan."},
    {"id": 2, "quarter": "Q1 2024", "topic": "migration",
     "text": "Q1 2024 migration retrospective: the alpha cutover paid back its $1.1M implementation cost inside 60 days. Customer NPS rose by 18 points and time-to-insight dropped from 6 hours to 9 minutes."},
    {"id": 3, "quarter": "Q2 2024", "topic": "migration",
     "text": "Migration project beta completed in Q2 2024 delivered $2.7M in incremental revenue. Beta included three large customer accounts that had been blocked on the alpha cutover."},
    {"id": 4, "quarter": "Q2 2024", "topic": "migration",
     "text": "Q2 2024 migration close-out report: beta cost $0.9M to ship and produced $2.7M in revenue and $0.6M in cost takeout. The legacy oracle license came off the books in Q3 2024."},
    {"id": 5, "quarter": "Q3 2024", "topic": "migration",
     "text": "Migration project gamma completed in Q3 2024 delivered $5.8M in incremental revenue. Gamma was the international rollout; EMEA and APAC came online together."},
    {"id": 6, "quarter": "Q3 2023", "topic": "migration",
     "text": "Q3 2023 migration to the new ETL stack produced $1.3M in incremental revenue, the smallest migration revenue figure of the program. Lessons learned fed the Q1 2024 alpha plan."},
    {"id": 7, "quarter": "Q1 2024", "topic": "platform",
     "text": "Q1 2024 platform reliability: 99.96% query availability, 11 incidents total, mean time to recovery 18 minutes."},
    {"id": 8, "quarter": "Q2 2024", "topic": "platform",
     "text": "Q2 2024 platform reliability: 99.93% query availability, 14 incidents, mean time to recovery 23 minutes; the regression was traced to the beta cutover."},
    {"id": 9, "quarter": "Q3 2024", "topic": "platform",
     "text": "Q3 2024 platform reliability: 99.98% query availability, 8 incidents, mean time to recovery 12 minutes. Best quarter in the program."},
    {"id": 10, "quarter": "Q1 2024", "topic": "ops",
     "text": "Q1 2024 operations: headcount steady at 42, one VP departure backfilled, two L4 promotions."},
    {"id": 11, "quarter": "Q2 2024", "topic": "ops",
     "text": "Q2 2024 operations: headcount up to 47, two new platform engineers, one principal architect hire."},
    {"id": 12, "quarter": "Q3 2024", "topic": "ops",
     "text": "Q3 2024 operations: headcount up to 51, the on-call rotation expanded to a 1-in-5 cadence."},
]

print(f"Corpus loaded: {len(CORPUS)} chunks across "
      f"{len(set(c['quarter'] for c in CORPUS))} quarters and "
      f"{len(set(c['topic'] for c in CORPUS))} topics.")


## Provision the retrieval substrate

Same shape as Lab 02. The next cell stands up the OpenSearch Serverless collection with its three policies, indexes every chunk, creates the Supabase pgvector table, and writes the 2-hour watchdog Lambda + EventBridge rule. The watchdog is the safety net: if the cleanup cell never runs (kernel dies, browser closes, the student walks away), the Lambda fires after 2 hours and destroys the collection. OpenSearch Serverless is the only resource in this lab that bills hourly.

OCU minimum is 4 (2 indexing + 2 search). Once the cell completes you are paying $0.96/hour against the wall clock until the collection is gone.

In [ ]:
# NBVAL_SKIP
# Provision OpenSearch Serverless + Supabase pgvector + watchdog Lambda.
# Idempotent: every create wraps "already exists" errors so re-running this
# cell after a prior partial setup picks up where it left off.

from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

aoss = boto3.client(
    "opensearchserverless",
    region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
events = boto3.client(
    "events",
    region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
lambda_client = boto3.client(
    "lambda",
    region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)


# 1. Three OpenSearch Serverless policies.
def _put_policy(name, policy_type, body):
    if policy_type in ("encryption", "network"):
        fn = aoss.create_security_policy
    else:
        fn = aoss.create_access_policy
    try:
        fn(name=name, type=policy_type, policy=json.dumps(body))
        print(f"  + {policy_type} policy {name} created")
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            print(f"  = {policy_type} policy {name} already exists")
        else:
            raise


_put_policy(
    ENCRYPTION_POLICY_NAME,
    "encryption",
    {"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}], "AWSOwnedKey": True},
)
_put_policy(
    NETWORK_POLICY_NAME,
    "network",
    [{"Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
        {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]},
    ], "AllowFromPublic": True}],
)
_put_policy(
    DATA_ACCESS_POLICY_NAME,
    "data",
    [{"Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"], "Permission": ["aoss:*"]},
        {"ResourceType": "index", "Resource": [f"index/{COLLECTION_NAME}/*"], "Permission": ["aoss:*"]},
    ], "Principal": [f"arn:aws:iam::{ACCOUNT_ID}:root"]}],
)


# 2. Create or fetch the collection. Wait for ACTIVE.
try:
    aoss.create_collection(name=COLLECTION_NAME, type="SEARCH", tags=[
        {"key": "labrun:lab-slug", "value": LAB_TAG_VALUE},
    ])
    print(f"  + collection {COLLECTION_NAME} requested")
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print(f"  = collection {COLLECTION_NAME} already requested")
    else:
        raise

# Wait for ACTIVE (typical 60-180s; ceiling 8 minutes).
print("Waiting for OpenSearch Serverless collection to reach ACTIVE...")
COLLECTION_ENDPOINT = None
COLLECTION_ID = None
for attempt in range(48):
    summaries = aoss.list_collections(collectionFilters={"name": COLLECTION_NAME}).get("collectionSummaries", [])
    if summaries:
        details = aoss.batch_get_collection(ids=[summaries[0]["id"]]).get("collectionDetails", [])
        if details:
            status = details[0].get("status")
            if status == "ACTIVE":
                COLLECTION_ID = details[0]["id"]
                COLLECTION_ENDPOINT = details[0]["collectionEndpoint"]
                print(f"  > collection ACTIVE: {COLLECTION_ENDPOINT}")
                break
            print(f"  ... still {status}, retrying ({attempt+1}/48)")
    time.sleep(10)
if COLLECTION_ENDPOINT is None:
    print("Collection did not reach ACTIVE within 8 minutes. Check AOSS console.")
    raise SystemExit(1)


# 3. Index every chunk into OpenSearch (BM25) and Supabase pgvector (dense).
host = COLLECTION_ENDPOINT.replace("https://", "")
awsauth = AWS4Auth(AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, REGION, "aoss")
os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)

try:
    os_client.indices.create(
        index=INDEX_NAME,
        body={"settings": {"index": {"knn": False}},
              "mappings": {"properties": {
                  "chunk_id": {"type": "integer"},
                  "text": {"type": "text"},
                  "quarter": {"type": "keyword"},
                  "topic": {"type": "keyword"},
              }}},
    )
    print(f"  + index {INDEX_NAME} created")
except Exception as exc:
    msg = str(exc)
    if "resource_already_exists" in msg or "already exists" in msg:
        print(f"  = index {INDEX_NAME} already exists")
    else:
        print(f"index create error (continuing): {exc!r}")

# Bulk index chunks. Refresh after.
for c in CORPUS:
    os_client.index(index=INDEX_NAME, id=str(c["id"]), body={
        "chunk_id": c["id"], "text": c["text"],
        "quarter": c["quarter"], "topic": c["topic"],
    })
os_client.indices.refresh(index=INDEX_NAME)
print(f"  > {len(CORPUS)} chunks indexed into OpenSearch")


# 4. Supabase pgvector table + dense vector population.
conn = psycopg2.connect(DB_URI)
conn.autocommit = True
try:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS public.{PGVECTOR_TABLE} (
                chunk_id integer PRIMARY KEY,
                quarter text,
                topic text,
                text text NOT NULL,
                embedding vector({EMBEDDING_DIM})
            )
        """)
        cur.execute(f"TRUNCATE TABLE public.{PGVECTOR_TABLE}")
finally:
    conn.close()

openai_client = OpenAI(api_key=OPENAI_API_KEY)


def embed(texts):
    resp = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [d.embedding for d in resp.data]


texts = [c["text"] for c in CORPUS]
vectors = embed(texts)

conn = psycopg2.connect(DB_URI)
conn.autocommit = True
try:
    with conn.cursor() as cur:
        for c, v in zip(CORPUS, vectors):
            cur.execute(
                f"INSERT INTO public.{PGVECTOR_TABLE} (chunk_id, quarter, topic, text, embedding) "
                f"VALUES (%s, %s, %s, %s, %s)",
                (c["id"], c["quarter"], c["topic"], c["text"], str(v)),
            )
finally:
    conn.close()
print(f"  > {len(CORPUS)} chunks embedded and inserted into pgvector")


# 5. Two-hour watchdog: IAM role, Lambda, EventBridge schedule.
ASSUME_ROLE_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"},
                   "Action": "sts:AssumeRole"}],
})
INLINE_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow",
         "Action": ["aoss:DeleteCollection", "aoss:ListCollections", "aoss:BatchGetCollection"],
         "Resource": "*"},
        {"Effect": "Allow",
         "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
         "Resource": "*"},
    ],
})

try:
    iam.create_role(
        RoleName=WATCHDOG_ROLE_NAME,
        AssumeRolePolicyDocument=ASSUME_ROLE_POLICY,
        Tags=[{"Key": "labrun:lab-slug", "Value": LAB_TAG_VALUE}],
    )
    iam.put_role_policy(
        RoleName=WATCHDOG_ROLE_NAME,
        PolicyName="inline",
        PolicyDocument=INLINE_POLICY,
    )
    print(f"  + watchdog role {WATCHDOG_ROLE_NAME} created")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"  = watchdog role {WATCHDOG_ROLE_NAME} already exists")
    else:
        raise

role_arn = iam.get_role(RoleName=WATCHDOG_ROLE_NAME)["Role"]["Arn"]

# IAM propagation: wait up to 30s for the role to be assumable by Lambda.
time.sleep(15)

LAMBDA_CODE = (
    "import boto3\n"
    "def handler(event, context):\n"
    "    aoss = boto3.client('opensearchserverless')\n"
    "    name = '" + COLLECTION_NAME + "'\n"
    "    cs = aoss.list_collections(collectionFilters={'name': name}).get('collectionSummaries', [])\n"
    "    for c in cs:\n"
    "        aoss.delete_collection(id=c['id'])\n"
    "    return {'deleted': len(cs)}\n"
)
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", LAMBDA_CODE)
zip_bytes = zip_buf.getvalue()

try:
    lambda_client.create_function(
        FunctionName=WATCHDOG_LAMBDA_NAME,
        Runtime="python3.11",
        Role=role_arn,
        Handler="lambda_function.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=120,
        Tags={"labrun:lab-slug": LAB_TAG_VALUE},
    )
    print(f"  + watchdog Lambda {WATCHDOG_LAMBDA_NAME} created")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceConflictException":
        print(f"  = watchdog Lambda {WATCHDOG_LAMBDA_NAME} already exists; updating code")
        lambda_client.update_function_code(FunctionName=WATCHDOG_LAMBDA_NAME, ZipFile=zip_bytes)
    else:
        raise

lambda_arn = lambda_client.get_function(FunctionName=WATCHDOG_LAMBDA_NAME)["Configuration"]["FunctionArn"]

# EventBridge rule: 2-hour scheduled trigger.
try:
    events.put_rule(
        Name=WATCHDOG_RULE_NAME,
        ScheduleExpression="rate(2 hours)",
        State="ENABLED",
        Tags=[{"Key": "labrun:lab-slug", "Value": LAB_TAG_VALUE}],
    )
    print(f"  + watchdog rule {WATCHDOG_RULE_NAME} created")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
        print(f"  = watchdog rule {WATCHDOG_RULE_NAME} already exists")
    else:
        raise

try:
    lambda_client.add_permission(
        FunctionName=WATCHDOG_LAMBDA_NAME,
        StatementId="EventBridgeInvoke",
        Action="lambda:InvokeFunction",
        Principal="events.amazonaws.com",
        SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{WATCHDOG_RULE_NAME}",
    )
except ClientError as e:
    if e.response["Error"]["Code"] not in ("ResourceConflictException",):
        raise

events.put_targets(Rule=WATCHDOG_RULE_NAME, Targets=[{"Id": "1", "Arn": lambda_arn}])
print("  > 2-hour watchdog wired up. If cleanup never runs, the collection auto-destructs in 2 hours.")

# Initialize the trace log fresh.
with open(TRACE_PATH, "w") as f:
    json.dump({"sessions": []}, f)
print(f"  > Trace log initialized at {TRACE_PATH}")

print()
print("Provisioning complete. The retrieval substrate is live.")


**Wallet check.** Provisioning landed. OpenSearch is on the clock at $0.96/OCU-hour against 4 OCU minimum. Embeddings: ~$0.001 on a corpus this small. Setup ping calls on Claude Haiku and OpenAI models.list: a fraction of a cent. Spent so far: about $0.05. The biggest line item is the OpenSearch hourly bill; the rest of the lab spends roughly equal Claude Sonnet tokens for the planner + composer loop on every test query.

## Task 1: Confirm the retrieval substrate is up

Same shape as Lab 02 Checkpoint 2 combined: the OpenSearch collection must be ACTIVE and the chunks index doc count must match the pgvector row count within tolerance 1.

You do not need to write code for Task 1. The provisioning cell above did the work. The next cell records the substrate state into the trace log so the validator can read it, and then calls Checkpoint 1.

In [ ]:
# NBVAL_SKIP
# Task 1: snapshot the substrate state into the trace log for Checkpoint 1.

def _trace_load():
    try:
        with open(TRACE_PATH, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {"sessions": []}


def _trace_write(data):
    with open(TRACE_PATH, "w") as f:
        json.dump(data, f, indent=2)


# YOUR CODE: query the collection status via aoss.batch_get_collection
# YOUR CODE: query the OpenSearch doc count for the chunks index
# YOUR CODE: query the pgvector row count from PGVECTOR_TABLE
# YOUR CODE: write a "substrate" block into the trace log with three fields:
# YOUR CODE:   collection_status, opensearch_doc_count, pgvector_row_count

# Default placeholders so the checkpoint surfaces a useful error_reason
# if the student leaves the cell empty.
collection_status = "UNKNOWN"
opensearch_doc_count = 0
pgvector_row_count = 0

trace = _trace_load()
trace["substrate"] = {
    "collection_status": collection_status,
    "opensearch_doc_count": opensearch_doc_count,
    "pgvector_row_count": pgvector_row_count,
}
_trace_write(trace)
print(f"Substrate snapshot recorded: status={collection_status}, "
      f"opensearch={opensearch_doc_count}, pgvector={pgvector_row_count}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: collection ACTIVE, doc count >= len(CORPUS), pgvector row diff in [0, 1].


def checkpoint_1(session):
    try:
        summaries = aoss.list_collections(collectionFilters={"name": COLLECTION_NAME}).get("collectionSummaries", [])
        if not summaries:
            return CheckpointResult(
                status="fail",
                error_reason=f"Collection {COLLECTION_NAME!r} not found in OpenSearch Serverless.",
            )
        details = aoss.batch_get_collection(ids=[summaries[0]["id"]]).get("collectionDetails", [])
        status = details[0].get("status") if details else None
        if status != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=f"Collection {COLLECTION_NAME!r} status is {status!r}; expected ACTIVE.",
            )

        os_count = os_client.count(index=INDEX_NAME).get("count", 0)
        if os_count < len(CORPUS):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OpenSearch index {INDEX_NAME!r} has {os_count} docs; expected at least "
                    f"{len(CORPUS)}. Re-run the provisioning cell."
                ),
            )

        conn = psycopg2.connect(DB_URI)
        try:
            with conn.cursor() as cur:
                cur.execute(f"SELECT COUNT(*) FROM public.{PGVECTOR_TABLE}")
                pg_count = cur.fetchone()[0]
        finally:
            conn.close()

        if abs(os_count - pg_count) > 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OpenSearch doc count ({os_count}) and pgvector row count ({pg_count}) "
                    f"differ by more than 1. The two stores must be in sync; check the bulk "
                    f"embed + insert step in the provisioning cell."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=repr(exc))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The provisioning cell already created and populated both stores. If Checkpoint 1 fails, the most likely cause is that the provisioning cell did not finish: the collection is still in `CREATING`, or the bulk embed step crashed before all 12 inserts landed. Re-run the provisioning cell and confirm every step printed a `>` line.

</details>

<details><summary>Hint 2 (direction)</summary>

The validator runs three live queries:

1. `aoss.batch_get_collection(ids=[collection_id])` to confirm `status == "ACTIVE"`.
2. `os_client.count(index=INDEX_NAME)` to read the OpenSearch doc count.
3. `SELECT COUNT(*) FROM public.{PGVECTOR_TABLE}` to read the pgvector row count.

If the two counts differ by more than 1, the bulk embed step crashed partway through. The fix is to drop the table, re-run provisioning, and confirm the print of `> 12 chunks embedded and inserted into pgvector` lands.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If Task 1 is failing because the snapshot block is empty, fill it in with live queries:

```python
summaries = aoss.list_collections(collectionFilters={"name": COLLECTION_NAME}).get("collectionSummaries", [])
details = aoss.batch_get_collection(ids=[summaries[0]["id"]]).get("collectionDetails", [])
collection_status = details[0].get("status") if details else "MISSING"
opensearch_doc_count = os_client.count(index=INDEX_NAME).get("count", 0)

conn = psycopg2.connect(DB_URI)
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM public.{PGVECTOR_TABLE}")
    pgvector_row_count = cur.fetchone()[0]
conn.close()
```

The substrate block in the trace log is informational; Checkpoint 1 reads live cloud state, not the trace.

</details>

**Wallet check.** Substrate is up. OpenSearch keeps ticking at $0.96/OCU-hour. Spent so far: still about $0.05. The next three checkpoints all share the same retrieval substrate, so no new infra cost until cleanup.

## Task 2: Planner step with Claude tool use

Claude Sonnet 4.6 ignores prompts that say "return JSON" about 5% of the time and free-texts an answer instead. The fix is Anthropic tool use: define a `submit_plan` tool with a strict JSON schema for `sub_queries`, then force the model to call it via `tool_choice={"type": "tool", "name": "submit_plan"}`. The model has no other path; it must emit a `tool_use` block, and the block's `input` field parses cleanly into your Pydantic model.

Write `plan_query(user_query)` that:

1. Calls `claude-sonnet-4-6` with the `submit_plan` tool and `tool_choice` forcing the tool.
2. Extracts the `tool_use` block from the response and parses its `input` into a `Plan` Pydantic model.
3. Returns the `Plan`.
4. Appends a record to the trace log with the user query, the parsed plan, and the raw response stop_reason.

The schema requires 2 to 5 sub-queries. The model handles the decomposition.

In [ ]:
# NBVAL_SKIP
# Task 2: the planner. Pydantic types + the submit_plan tool schema are pre-wired;
# the student fills in plan_query() to call Claude with tool_choice forced.

from pydantic import BaseModel, Field, ValidationError


class SubQuery(BaseModel):
    question: str = Field(min_length=1)
    rationale: str = Field(min_length=1)


class Plan(BaseModel):
    sub_queries: list[SubQuery] = Field(min_length=2, max_length=5)


SUBMIT_PLAN_TOOL = {
    "name": "submit_plan",
    "description": (
        "Submit a query decomposition plan. The user's question must be broken into "
        "2 to 5 self-contained sub-questions that each describe a single retrieval "
        "the system should perform."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "sub_queries": {
                "type": "array",
                "minItems": 2,
                "maxItems": 5,
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "minLength": 1,
                                     "description": "A self-contained sub-question."},
                        "rationale": {"type": "string", "minLength": 1,
                                      "description": "Why this sub-question is needed."},
                    },
                    "required": ["question", "rationale"],
                },
            },
        },
        "required": ["sub_queries"],
    },
}

PLANNER_SYSTEM_PROMPT = (
    "You are a query planner for an enterprise RAG system. The retrieval corpus is small "
    "and contains documents tagged by quarter (Q1 2024, Q2 2024, Q3 2024, Q3 2023) and by "
    "topic (migration, platform, ops). The user's question often spans multiple quarters "
    "or topics. Decompose it into 2 to 5 self-contained sub-questions that each map to a "
    "single retrieval. Always call the submit_plan tool. Never reply with free text."
)


def plan_query(user_query):
    """Decompose user_query into 2-5 sub-queries via forced tool use."""
    # YOUR CODE: call anthropic_client.messages.create with model=PLANNER_MODEL
    # YOUR CODE:   system=PLANNER_SYSTEM_PROMPT, tools=[SUBMIT_PLAN_TOOL]
    # YOUR CODE:   tool_choice={"type": "tool", "name": "submit_plan"}
    # YOUR CODE:   messages=[{"role": "user", "content": user_query}]
    # YOUR CODE:   max_tokens=1024
    # YOUR CODE: scan response.content for the tool_use block (block.type == "tool_use")
    # YOUR CODE: parse block.input into a Plan via Plan.model_validate
    # YOUR CODE: append a record to the trace log under "plans" with user_query,
    # YOUR CODE:   the plan (plan.model_dump()), and response.stop_reason
    # YOUR CODE: return the Plan
    raise NotImplementedError("Fill in plan_query for Task 2.")


anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Run the planner against the lab's test query and print the result so the
# operator can see it before the checkpoint runs.
TEST_PLAN = plan_query(TEST_QUERY)
print(f"Plan for test query produced {len(TEST_PLAN.sub_queries)} sub-queries:")
for i, sq in enumerate(TEST_PLAN.sub_queries):
    print(f"  {i+1}. {sq.question}")
    print(f"     rationale: {sq.rationale}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: planner produces a structured plan with 2-5 sub-queries; the
# underlying call used Anthropic tool use, not free text.


def checkpoint_2(session):
    if not isinstance(TEST_PLAN, Plan):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"plan_query did not return a Plan; got {type(TEST_PLAN).__name__}. "
                f"Confirm the function parses the tool_use input via Plan.model_validate."
            ),
        )
    if not (2 <= len(TEST_PLAN.sub_queries) <= 5):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Plan has {len(TEST_PLAN.sub_queries)} sub-queries; expected 2 to 5. "
                f"Most common cause: the planner returned a single sub-query identical "
                f"to the user query. Confirm the tool's input_schema has minItems: 2 and "
                f"maxItems: 5 on sub_queries, and that tool_choice forces submit_plan."
            ),
        )
    for i, sq in enumerate(TEST_PLAN.sub_queries):
        if not sq.question or not sq.rationale:
            return CheckpointResult(
                status="fail",
                error_reason=f"Sub-query {i+1} has an empty question or rationale.",
            )

    trace = _trace_load()
    plans = trace.get("plans", [])
    if not plans:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "The trace log has no 'plans' entry. Confirm plan_query appends a record "
                "to the trace log so the validator can read the underlying API response."
            ),
        )
    latest = plans[-1]
    if latest.get("stop_reason") != "tool_use":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Trace 'plans' entry has stop_reason={latest.get('stop_reason')!r}; "
                f"expected 'tool_use'. The planner returned free text instead of calling "
                f"the submit_plan tool. Set tool_choice={{'type': 'tool', 'name': 'submit_plan'}} "
                f"on the messages.create call."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

If the plan came back as a single-element list or as free text, the planner is not actually using the tool. Anthropic models prefer free text by default. You have to force the tool with `tool_choice`.

</details>

<details><summary>Hint 2 (direction)</summary>

On the `messages.create` call, set `tool_choice={"type": "tool", "name": "submit_plan"}`. This forces the model to emit a `tool_use` block instead of free text.

After the call returns, walk `response.content`:

```
for block in response.content:
    if block.type == "tool_use":
        plan = Plan.model_validate(block.input)
        break
```

Confirm your `SUBMIT_PLAN_TOOL` schema requires `sub_queries` to be an array with `minItems: 2`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working `plan_query` body:

```python
def plan_query(user_query):
    response = anthropic_client.messages.create(
        model=PLANNER_MODEL,
        max_tokens=1024,
        system=PLANNER_SYSTEM_PROMPT,
        tools=[SUBMIT_PLAN_TOOL],
        tool_choice={"type": "tool", "name": "submit_plan"},
        messages=[{"role": "user", "content": user_query}],
    )

    plan = None
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_plan":
            plan = Plan.model_validate(block.input)
            break
    if plan is None:
        raise RuntimeError("Planner returned no tool_use block")

    trace = _trace_load()
    plans = trace.setdefault("plans", [])
    plans.append({
        "user_query": user_query,
        "plan": plan.model_dump(),
        "stop_reason": response.stop_reason,
    })
    _trace_write(trace)
    return plan
```

</details>

**Wallet check.** Planner landed. One Claude Sonnet call: ~500 input tokens + ~250 output tokens, about a quarter of a cent. Spent so far: about $0.06.

## Task 3: Executor step

The executor iterates over `plan.sub_queries` and runs the Lab 02 hybrid retrieval pipeline against each one. For this lab the executor uses just the dense path (pgvector cosine similarity); BM25 is available on the OpenSearch side if you want to extend the retrieval, but the eval set is small enough that dense alone hits the multi-hop hooks.

Write `execute_plan(plan)` that, for each sub-query:

1. Embeds the sub-query via `text-embedding-3-small`.
2. Runs a cosine similarity search against the pgvector table, returning the top 5 chunks with their cosine similarity scores.
3. Records latency in milliseconds.
4. Appends an `executor_results` block to the trace log with one entry per sub-query: `sub_query`, `top_5`, `retrieval_latency_ms`.

The classic bug here is an accumulator that overwrites entries by key. Use a list of dicts, one per sub-query, in plan order.

In [ ]:
# NBVAL_SKIP
# Task 3: the executor. Helper functions for embedding + pgvector search are
# pre-wired; the student fills in the execute_plan loop.


def embed_one(text):
    return embed([text])[0]


def dense_search(query_vec, top_k=5):
    """pgvector cosine similarity search. Returns list of dicts:
    {'chunk_id': int, 'text': str, 'quarter': str, 'topic': str, 'score': float}.
    score is cosine similarity (1.0 = identical), computed as 1 - cosine_distance."""
    vec_str = str(query_vec)
    conn = psycopg2.connect(DB_URI)
    try:
        with conn.cursor() as cur:
            cur.execute(
                f"SELECT chunk_id, text, quarter, topic, "
                f"1 - (embedding <=> %s::vector) AS score "
                f"FROM public.{PGVECTOR_TABLE} "
                f"ORDER BY embedding <=> %s::vector "
                f"LIMIT %s",
                (vec_str, vec_str, top_k),
            )
            rows = cur.fetchall()
    finally:
        conn.close()
    return [{"chunk_id": r[0], "text": r[1], "quarter": r[2], "topic": r[3],
             "score": float(r[4])} for r in rows]


def execute_plan(plan):
    """Run each sub-query through dense retrieval. Returns a list of result dicts,
    one per sub-query in plan order. Also appends to the trace log under
    'executor_results'."""
    # YOUR CODE: initialize an empty list `results`
    # YOUR CODE: iterate over plan.sub_queries (each sq has .question and .rationale)
    # YOUR CODE:   record t0 = time.time()
    # YOUR CODE:   vec = embed_one(sq.question)
    # YOUR CODE:   top_5 = dense_search(vec, top_k=5)
    # YOUR CODE:   latency_ms = int((time.time() - t0) * 1000)
    # YOUR CODE:   append a dict {sub_query, top_5, retrieval_latency_ms} to results
    # YOUR CODE: append results to the trace log under "executor_results" (a list of runs)
    # YOUR CODE: return results
    raise NotImplementedError("Fill in execute_plan for Task 3.")


TEST_EXECUTOR_RESULTS = execute_plan(TEST_PLAN)
print(f"Executor returned {len(TEST_EXECUTOR_RESULTS)} sub-query result blocks.")
for i, r in enumerate(TEST_EXECUTOR_RESULTS):
    top = r["top_5"][0] if r["top_5"] else {"chunk_id": "?", "score": 0.0}
    print(f"  {i+1}. {r['sub_query'][:70]} ...")
    print(f"     top chunk: id={top['chunk_id']} score={top['score']:.3f} "
          f"latency={r['retrieval_latency_ms']}ms")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: executor results length matches plan length, every entry has
# all three fields, at least 80% have non-empty top_5.


def checkpoint_3(session):
    trace = _trace_load()
    runs = trace.get("executor_results", [])
    if not runs:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Trace 'executor_results' is empty. Confirm execute_plan appends a "
                "block to the trace log after the loop."
            ),
        )
    latest = runs[-1]
    if len(latest) != len(TEST_PLAN.sub_queries):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Executor produced {len(latest)} result entries for a plan with "
                f"{len(TEST_PLAN.sub_queries)} sub-queries. Most common bug: the "
                f"accumulator is a dict keyed by sub-query text and a duplicate text "
                f"overwrites the prior entry. Use a list, not a dict."
            ),
        )
    required_fields = {"sub_query", "top_5", "retrieval_latency_ms"}
    for i, entry in enumerate(latest):
        missing = required_fields - set(entry.keys())
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=f"Executor entry {i} missing fields: {sorted(missing)}",
            )
    nonempty = sum(1 for e in latest if e["top_5"])
    if nonempty / len(latest) < 0.8:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Only {nonempty}/{len(latest)} sub-queries returned non-empty top_5 lists. "
                f"Confirm dense_search is connected to the populated pgvector table."
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

If the executor result count does not match the plan sub-query count, the loop is not iterating over every sub-query. Print `len(plan.sub_queries)` and `len(results)` before returning; they must match.

</details>

<details><summary>Hint 2 (direction)</summary>

Common bugs:

1. The accumulator is a dict keyed by `sq.question`. If two sub-queries happen to have similar text, the dict collapses them. Use a list.
2. The trace write happens inside the loop and overwrites a prior key. Append to a list at the end of the loop, not in the middle.
3. Latency is captured but in seconds rather than milliseconds. Multiply by 1000 and cast to int.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working `execute_plan` body:

```python
def execute_plan(plan):
    results = []
    for sq in plan.sub_queries:
        t0 = time.time()
        vec = embed_one(sq.question)
        top_5 = dense_search(vec, top_k=5)
        latency_ms = int((time.time() - t0) * 1000)
        results.append({
            "sub_query": sq.question,
            "top_5": top_5,
            "retrieval_latency_ms": latency_ms,
        })

    trace = _trace_load()
    runs = trace.setdefault("executor_results", [])
    runs.append(results)
    _trace_write(trace)
    return results
```

</details>

**Wallet check.** Executor ran three or four embed calls plus pgvector queries. Embeddings on chunks this small cost a fraction of a cent. pgvector is free tier. Spent so far: about $0.07.

## Task 4: Composer step with citations

The composer takes the plan + executor results and writes the final answer. Two things must hold:

1. Every claim is followed by a citation of the form `[chunk_<id>]`, where `<id>` is the integer chunk ID from the retrieved context.
2. Every sub-query in the plan is represented by at least one citation in the answer.

The trap is that Claude Sonnet writes confident, beautifully-worded paragraphs by default and will ignore your citation instructions if they are buried in a long system prompt. The fix is to put the citation rule first and make the user message render every chunk with its ID prefix: `[chunk_42]: <text>`.

Write `compose_answer(user_query, plan, executor_results)` that returns the final answer string and appends an `answers` block to the trace.

In [ ]:
# NBVAL_SKIP
# Task 4: the composer. The system prompt + chunk-rendering helper are
# pre-wired; the student fills in compose_answer.

COMPOSER_SYSTEM_PROMPT = (
    "You are a careful enterprise analyst. Every factual claim in your answer must "
    "be followed by a citation of the form `[chunk_N]` where N is the integer chunk "
    "ID from the retrieved context provided in the user message. Use one citation "
    "per claim. Do not invent chunk IDs. Do not cite chunks that were not in the "
    "retrieved context. If the retrieved context does not support a claim, omit the "
    "claim. Write 2 to 4 short paragraphs, total. No preamble."
)


def render_chunks_for_composer(executor_results):
    """Render every retrieved chunk into a single context block prefixed by its ID.
    Deduplicates by chunk_id so the composer does not see the same chunk twice."""
    seen = set()
    lines = []
    for r in executor_results:
        for c in r["top_5"]:
            cid = c["chunk_id"]
            if cid in seen:
                continue
            seen.add(cid)
            lines.append(f"[chunk_{cid}]: {c['text']}")
    return "\n\n".join(lines)


def compose_answer(user_query, plan, executor_results):
    """Compose a final answer grounded in retrieved chunks. Returns the answer
    string and appends to the trace log under 'answers'."""
    context_block = render_chunks_for_composer(executor_results)
    user_message = (
        f"User question: {user_query}\n\n"
        f"Retrieved context (each entry begins with its chunk ID):\n\n"
        f"{context_block}\n\n"
        f"Answer the question. Cite every claim with [chunk_N] using only the IDs above."
    )

    # YOUR CODE: call anthropic_client.messages.create with model=COMPOSER_MODEL
    # YOUR CODE:   system=COMPOSER_SYSTEM_PROMPT, max_tokens=1024
    # YOUR CODE:   messages=[{"role": "user", "content": user_message}]
    # YOUR CODE: extract the answer text from response.content[0].text
    # YOUR CODE: append to the trace log under "answers": a list of
    # YOUR CODE:   {user_query, answer, plan_subqueries: [sq.question for sq in plan.sub_queries]}
    # YOUR CODE: return the answer string

    raise NotImplementedError("Fill in compose_answer for Task 4.")


TEST_ANSWER = compose_answer(TEST_QUERY, TEST_PLAN, TEST_EXECUTOR_RESULTS)
print("Composer answer:")
print("-" * 60)
print(TEST_ANSWER)
print("-" * 60)


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: composer answer has at least one [chunk_N] citation per
# sub-query, and every cited chunk is in the executor's retrieved set.


CITATION_RE = re.compile(r"\[chunk_(\d+)\]")


def checkpoint_4(session):
    if not isinstance(TEST_ANSWER, str) or not TEST_ANSWER.strip():
        return CheckpointResult(
            status="fail",
            error_reason="compose_answer returned an empty or non-string answer.",
        )
    citations = CITATION_RE.findall(TEST_ANSWER)
    if len(citations) < len(TEST_PLAN.sub_queries):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Answer contains {len(citations)} chunk citations; expected at least "
                f"{len(TEST_PLAN.sub_queries)} (one per sub-query). Confirm the composer "
                f"system prompt instructs citation in the form [chunk_N] AND the user "
                f"message prefixes each retrieved chunk with its [chunk_N]: label."
            ),
        )
    cited_ids = {int(c) for c in citations}
    retrieved_ids = set()
    for r in TEST_EXECUTOR_RESULTS:
        for c in r["top_5"]:
            retrieved_ids.add(c["chunk_id"])
    invalid = cited_ids - retrieved_ids
    if invalid:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Answer cites chunk IDs {sorted(invalid)} that were not in the retrieved "
                f"context. Most common cause: the chunk-rendering step in the composer's "
                f"user message rendered the dict object instead of the integer id field."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

If the final answer has zero citations or has citations that do not match real chunk IDs, the system prompt or the chunk-rendering step is the suspect. Print the user message you are sending to Claude and inspect the chunk-id prefixes by eye.

</details>

<details><summary>Hint 2 (direction)</summary>

The composer's system prompt must include an explicit instruction like "Every factual claim in your answer must be followed by a citation of the form `[chunk_N]` where N is the chunk ID from the retrieved context." (the pre-wired `COMPOSER_SYSTEM_PROMPT` already says this).

When you render the retrieved chunks into the user message, prefix each chunk with its integer ID in the canonical form: `[chunk_42]: <text>`. The pre-wired `render_chunks_for_composer` already does this.

The remaining job for `compose_answer` is calling the model and reading `response.content[0].text` back out.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working `compose_answer` body:

```python
def compose_answer(user_query, plan, executor_results):
    context_block = render_chunks_for_composer(executor_results)
    user_message = (
        f"User question: {user_query}\n\n"
        f"Retrieved context (each entry begins with its chunk ID):\n\n"
        f"{context_block}\n\n"
        f"Answer the question. Cite every claim with [chunk_N] using only the IDs above."
    )

    response = anthropic_client.messages.create(
        model=COMPOSER_MODEL,
        max_tokens=1024,
        system=COMPOSER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    answer = response.content[0].text

    trace = _trace_load()
    answers = trace.setdefault("answers", [])
    answers.append({
        "user_query": user_query,
        "answer": answer,
        "plan_subqueries": [sq.question for sq in plan.sub_queries],
    })
    _trace_write(trace)
    return answer
```

</details>

**Wallet check.** Planner + executor + composer = one full loop. Claude Sonnet composer call: ~1500 input tokens (the rendered context) + ~400 output tokens. About 1.4 cents. Spent so far: about $0.09 on API calls plus the OpenSearch hourly tick.

## Task 5: Refuse to answer on under-retrievable queries

The composer will confabulate if you let it. Five carefully-chosen impossible queries (board meetings in Reykjavik in 1958, puffin litigation, dragonfly logistics) live in `IMPOSSIBLE_QUERIES`. None of the 12 corpus chunks have anything to say about them; every sub-query's top-1 chunk should land well below the 0.3 cosine similarity threshold.

The guard runs between executor and composer. For each sub-query, look at the top-1 retrieval's `score` field. If ANY sub-query has top-1 `score < REFUSAL_THRESHOLD`, the composer is bypassed and the canonical refusal string is returned with a short "here is what I did find" tail that lists the top-1 chunk per sub-query.

Write `should_refuse(executor_results, threshold)` returning a bool, then `agent_run(user_query)` that orchestrates plan -> execute -> guard -> compose, returning either the canonical refusal or the composed answer. Append every run to the trace log under `agent_runs`.

The 5/5 refusal rate over `IMPOSSIBLE_QUERIES` is the graded checkpoint.

In [ ]:
# NBVAL_SKIP
# Task 5: refuse-to-answer guard + top-level agent orchestration.


def should_refuse(executor_results, threshold=REFUSAL_THRESHOLD):
    """Return True if ANY sub-query's top-1 chunk has score < threshold."""
    # YOUR CODE: iterate over executor_results
    # YOUR CODE:   if entry has empty top_5 -> return True
    # YOUR CODE:   if entry["top_5"][0]["score"] < threshold -> return True
    # YOUR CODE: return False
    raise NotImplementedError("Fill in should_refuse for Task 5.")


def build_refusal_message(executor_results):
    """Build the canonical refusal string with a tail of top-1 chunks per sub-query."""
    lines = [CANONICAL_REFUSAL_PREFIX + ":"]
    for r in executor_results:
        if r["top_5"]:
            top = r["top_5"][0]
            lines.append(
                f"  sub-query {r['sub_query'][:60]!r} -> top chunk_{top['chunk_id']} "
                f"(score {top['score']:.2f}): {top['text'][:120]}"
            )
        else:
            lines.append(f"  sub-query {r['sub_query'][:60]!r} -> no chunks retrieved")
    return "\n".join(lines)


def agent_run(user_query):
    """Top-level orchestration: plan -> execute -> refuse-or-compose.
    Returns the final answer string (either the canonical refusal or the
    composer output) and appends to trace under 'agent_runs'."""
    plan = plan_query(user_query)
    executor_results = execute_plan(plan)

    if should_refuse(executor_results, threshold=REFUSAL_THRESHOLD):
        answer = build_refusal_message(executor_results)
        path = "refused"
    else:
        answer = compose_answer(user_query, plan, executor_results)
        path = "composed"

    trace = _trace_load()
    runs = trace.setdefault("agent_runs", [])
    runs.append({
        "user_query": user_query,
        "path": path,
        "answer": answer,
        "plan_subqueries": [sq.question for sq in plan.sub_queries],
    })
    _trace_write(trace)
    return answer


# Run the agent against all 5 impossible queries.
print("Running 5 impossible queries through the agent. Each should refuse:")
print("=" * 60)
for i, q in enumerate(IMPOSSIBLE_QUERIES):
    print(f"[{i+1}/5] {q}")
    out = agent_run(q)
    print(f"  -> {'REFUSED' if out.startswith(CANONICAL_REFUSAL_PREFIX) else 'ANSWERED'}")
    print()


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: all 5 impossible queries hit the refusal path.


def checkpoint_5(session):
    trace = _trace_load()
    runs = trace.get("agent_runs", [])
    impossible_set = set(IMPOSSIBLE_QUERIES)
    matched = [r for r in runs if r.get("user_query") in impossible_set]
    if len(matched) < len(IMPOSSIBLE_QUERIES):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Trace 'agent_runs' has {len(matched)} entries matching the impossible "
                f"queries; expected {len(IMPOSSIBLE_QUERIES)}. Re-run the cell that loops "
                f"over IMPOSSIBLE_QUERIES."
            ),
        )
    # Keep the last 5 (in case the lab was re-run); these must all be refusals.
    last_runs = matched[-len(IMPOSSIBLE_QUERIES):]
    confabulated = [r for r in last_runs if r.get("path") != "refused"]
    if confabulated:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{len(confabulated)} of 5 impossible queries went down the composer path "
                f"instead of refusing. The refuse-to-answer guard runs between executor "
                f"and composer; if any sub-query's top-1 score is below 0.3 (cosine "
                f"similarity), the composer is bypassed and the canonical refusal string "
                f"is returned. Confirm should_refuse is checking score, not rank."
            ),
        )
    for r in last_runs:
        if not r.get("answer", "").startswith(CANONICAL_REFUSAL_PREFIX):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "An impossible-query run returned a refusal path but the answer string "
                    "does not start with the canonical refusal prefix. Confirm "
                    "build_refusal_message returns the canonical string verbatim."
                ),
            )
    return CheckpointResult(status="pass")


check(5, checkpoint_5)


<details><summary>Hint 1 (nudge)</summary>

If at least one impossible query went down the composer path, the guard is missing or its threshold check is wrong. Print the top-1 score for every sub-query on an impossible query; every one should be well below 0.3.

</details>

<details><summary>Hint 2 (direction)</summary>

The guard runs between executor and composer. For each sub-query, look at the top-1 retrieval's `score` field (not rank, not position). If any sub-query's top-1 score is below 0.3, skip the composer.

Do not use a Claude classifier for this. The raw cosine similarity score from the dense_search result is the right signal; a classifier adds latency and another point of failure.

The pre-wired `build_refusal_message` returns the canonical string. The student's job in `should_refuse` is a simple any-style check across the executor results.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working `should_refuse`:

```python
def should_refuse(executor_results, threshold=REFUSAL_THRESHOLD):
    for r in executor_results:
        if not r["top_5"]:
            return True
        if r["top_5"][0]["score"] < threshold:
            return True
    return False
```

The orchestration in `agent_run` is pre-wired; if Checkpoint 5 is still failing after fixing `should_refuse`, re-run the cell so all 5 impossible queries land fresh in the trace log.

</details>

**Wallet check.** Five impossible queries. Each one ran the planner (~500 in / ~250 out tokens) plus the executor (5 embeddings) plus the guard (no Claude call). The composer was bypassed all five times. Total Claude side: ~5 Sonnet planner calls, about 6 cents. Embeddings: still pennies. OpenSearch has been on the clock for roughly 30 to 60 minutes by now. Spent so far: about $0.60 to $1.10 depending on wall clock.

## Cleanup

Critical tier first: the OpenSearch Serverless collection (so the $0.96/OCU-hour clock stops), then its three policies, then the EventBridge rule + Lambda + IAM role for the 2-hour watchdog, then the Supabase pgvector table, then the local trace JSON.

The cleanup runs the three-phase protocol from RESOURCE-SAFETY-SPEC: teardown, verification scan, tag audit. If anything fails to delete, the cell prints the exact CLI command to run manually and exits with code 1.

In [ ]:
# NBVAL_SKIP
# Cleanup. Critical-first. OpenSearch Serverless deletes first; everything else
# is free at rest but cleared for hygiene.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(1 for r in CLEANUP_MANIFEST if r.critical and r.id in failed_ids)
standard_destroyed = standard_total - sum(1 for r in CLEANUP_MANIFEST if not r.critical and r.id in failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $1.80 to $2.50.** OpenSearch Serverless ran for roughly 90 minutes at $0.96/OCU-hour x 4 OCU = $1.20 to $1.60. Claude Sonnet planner + composer calls across 6 successful test queries (1 multi-hop + 5 impossible): about $0.40 to $0.60. OpenAI embeddings: pennies. Cleanup is done. Check your AWS Billing console in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. The planner step decomposes user queries into 2 to 5 sub-queries. Some user queries do not need decomposition at all ("how many incidents did we have in Q1 2024?" is a single retrieval). How would you change the planner's tool definition or its prompt to let it return a single sub-query without paying the round-trip cost of decomposition? Think about whether you change `minItems` on the schema, add a "skip planner" tool, or run a cheap upstream classifier on the user query.

2. The refuse-to-answer threshold (0.3 cosine similarity) is a magic number. What experiment would you run on your eval set to validate or tune this threshold? Be specific about the metric you would optimize (precision of refusals, recall of refusals, F1 against a labeled set of "answerable" vs "not answerable") and how you would label the eval set.

## Exam-Style Practice

**Question 1 (planner failure mode):**

An agentic RAG bot's planner is built on Claude Sonnet with a `submit_plan` tool defined. The team reports that 1 in 20 user queries lands a free-text reply that fails Pydantic parsing instead of a `tool_use` block. The bot crashes on those queries. Which change is the highest-leverage single fix?

A. Set `tool_choice={"type": "tool", "name": "submit_plan"}` on the `messages.create` call.
B. Switch the planner model from Claude Sonnet to Claude Opus.
C. Raise the planner's `max_tokens` from 1024 to 4096.
D. Add a retry loop that catches the parse error and re-calls Claude up to 3 times.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: `tool_choice` with `type: "tool"` forces the model to emit a `tool_use` block on every response. The 5% free-text rate goes to 0%. This is the canonical fix for "the model ignores my tool definition some of the time."
- B is wrong: Opus is more capable on hard reasoning but the default behavior for free text vs tool use is the same across the Sonnet/Opus tier. The fix is `tool_choice`, not model swap.
- C is wrong: max_tokens has no relationship to whether the model emits a tool_use block.
- D is wrong: a retry loop masks the problem and triples your bill on the 5% of queries that fail. Force the tool once at the API layer.

</details>

**Question 2 (refuse-to-answer guard placement):**

An agentic RAG pipeline has a planner -> executor -> composer flow. The product owner reports that the bot confabulates an answer about 10% of the time on questions whose retrievals returned mostly junk. Where in the pipeline does the refuse-to-answer guard belong, and what signal should it read?

A. Between executor and composer, reading the top-1 cosine similarity per sub-query.
B. Inside the composer's system prompt, asking the model to refuse if it is unsure.
C. Inside the planner, asking the model to predict whether the question is answerable.
D. After the composer, running a separate Claude Haiku call to grade the answer.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: the guard runs on a deterministic signal (cosine similarity from the retrieval result) at the only point in the pipeline where the bot has both the question and the retrieved evidence. Score-driven guards are reliable; LLM-driven guards add tokens and another point of failure.
- B is wrong: asking the composer to "refuse if unsure" trains the bot to be confident anyway. Refusal is not a vibes check; it is a thresholded check on a measurable signal.
- C is wrong: the planner has no retrieval signal yet. It cannot tell which questions are answerable until the executor runs.
- D is wrong: a post-hoc grading call ships an answer first, then maybe overturns it. By the time the grader runs, the confabulated answer is already in the user's hands; you have also doubled your token bill.

</details>

**Question 3 (citation grounding):**

A composer step grounds answers in retrieved chunks via `[chunk_N]` inline citations. The team reports that 12% of answers contain citations to chunk IDs that were not in the executor's retrieved set. What is the most likely root cause?

A. The chunk-rendering helper passes the dict object to the prompt template instead of the integer `chunk_id` field, so the model hallucinates plausible-looking IDs.
B. The composer model is too small; swap to a larger model.
C. The executor returned the wrong chunks; the retrieval is broken.
D. The system prompt does not mention citations at all.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: this is the classic Python-to-prompt bug. The composer sees something like `{'chunk_id': 42, 'text': '...'}` in the rendered context and writes `[chunk_object]` or makes up an integer. The fix is to render each chunk as `[chunk_42]: <text>` so the integer ID is unambiguous in the rendered text.
- B is wrong: model size is unrelated to whether the chunk IDs in the prompt are correctly formatted.
- C is wrong: if the executor returned the wrong chunks, the answer would cite the IDs of the wrong chunks correctly; the validator's check (cited IDs subset of retrieved IDs) would still pass.
- D is wrong: the question says the answers do contain `[chunk_N]` citations; the system prompt is being followed at the syntax level. The bug is the contents of N, not the format.

</details>